# M2 verified-source acquisition launcher
Thin launcher only. It records an exact repository commit, stages downloads locally, and mirrors only verified sources. Canonical Gate B remains blocked by the capture-design decision.

In [ ]:
from pathlib import Path
import os, platform, shutil, subprocess, sys
REPOSITORY_URL = "https://github.com/jcollins-bioinfo/giab-wes-nextflow.git"
REPOSITORY_REF = "main"  # Replace with an exact branch, tag, or commit for review runs.
REPOSITORY_DIR = Path("/content/giab-wes-nextflow")

def run_checked(command, cwd=None):
    result = subprocess.run(command, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout, end="")
    if result.returncode:
        raise subprocess.CalledProcessError(result.returncode, command, output=result.stdout)
    return result.stdout.strip()

if not REPOSITORY_DIR.exists():
    run_checked(["git", "clone", "--no-checkout", REPOSITORY_URL, str(REPOSITORY_DIR)])
else:
    if run_checked(["git", "status", "--porcelain"], REPOSITORY_DIR): raise RuntimeError("Refusing dirty checkout")
    origin=run_checked(["git", "remote", "get-url", "origin"], REPOSITORY_DIR).removesuffix(".git").lower()
    if origin != REPOSITORY_URL.removesuffix(".git").lower(): raise RuntimeError("Unrelated checkout")
run_checked(["git", "fetch", "origin", REPOSITORY_REF], REPOSITORY_DIR)
RESOLVED_SHA=run_checked(["git", "rev-parse", "FETCH_HEAD"], REPOSITORY_DIR)
run_checked(["git", "checkout", "--detach", RESOLVED_SHA], REPOSITORY_DIR)
print("requested_ref=", REPOSITORY_REF, "resolved_sha=", RESOLVED_SHA)
run_checked([sys.executable, "-m", "pip", "install", "--quiet", "--force-reinstall", str(REPOSITORY_DIR)])
import giab_wes_nextflow
print(giab_wes_nextflow.__version__, Path(giab_wes_nextflow.__file__).resolve())
run_checked([sys.executable, "-m", "giab_wes_nextflow.validation"])

## Private Drive and local staging
Mount the established private workspace. `/content/m2-stage` is ephemeral active I/O; Drive is only the durable verified-source mirror.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT=Path("/content/drive/MyDrive/giab-wes-nextflow-private")
STAGING=Path("/content/m2-stage")
if not DRIVE_ROOT.is_dir(): raise FileNotFoundError("Expected Drive workspace is absent")
if (DRIVE_ROOT/"DO NOT ACCESS WITH CHATGPT").exists(): raise PermissionError("Workspace safety marker present")
print("architecture",platform.machine(),"python",sys.version.split()[0],"memory/disk checks configured")
print("local_free",shutil.disk_usage("/content").free,"drive_free",shutil.disk_usage(DRIVE_ROOT).free)
for tool in ("git",):
    if not shutil.which(tool): raise RuntimeError(f"missing tool: {tool}")
run_checked(["giab-wes-acquire-m2", "--workspace", str(STAGING), "--preflight-only"])

## Acquire and mirror
Choose a new deliberate run ID. The helper prints combined sanitized output before raising.

In [ ]:
from datetime import datetime, timezone
RUN_ID = "m2-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
run_checked(["giab-wes-acquire-m2", "--workspace", str(STAGING), "--run-id", RUN_ID])
run_checked(["giab-wes-validate-m2", "--workspace", str(STAGING), "--run-id", RUN_ID])
run_checked(["giab-wes-mirror-m2", "--staging", str(STAGING), "--drive-root", str(DRIVE_ROOT), "--run-id", RUN_ID, "--repository-sha", RESOLVED_SHA])
print("Verified source mirror complete; this is not canonical Gate B.")
print("Next stage blocked until config/m2-target-design.json is confirmed with approved hashes.")

## Recovery
After a lost runtime, repeat bootstrap and Drive mount, set the prior `RUN_ID`, then hydrate and revalidate:

In [ ]:
# run_checked(["giab-wes-mirror-m2", "--hydrate", "--staging", str(STAGING), "--drive-root", str(DRIVE_ROOT), "--run-id", RUN_ID])